# 🎆 The Grand Finale — the whole play, performed live

Eleven notebooks ago this was a story: *Ada buys 50 Mbps from Bell for 10 TOK; one atomic
transaction mints ticket #7; a deterministic controller honors it; Bell's on-chain
revocation kills the session mid-window.* Since then you have met every component alone —
the shapes (02), the ports (03), the cardboard play (05), the chain (06), the signatures
(07), the bouncer (08), the hands (09), the brains (10), and the worlds that compose them
(11). Tonight they all go on stage **at the same time**, at the maximum realism a headless
notebook allows.

**What you'll be able to do after this notebook**

- perform the complete lifecycle — offer → judgment → atomic swap → challenge–response →
  activation → revocation — on a **real chain, with real signatures and the real
  controller**, and say which trust domain each step lives in
- point at the single judgment call and the two understudies, and give the exact recipe
  that upgrades each to fully real
- trace the revocation showpiece hop by hop: an ERC-721 flag flips on a blockchain, and a
  network session dies
- pass the integration exam spanning notebooks 00–12

**You need:** notebooks 00–11 · Foundry's `anvil` on PATH and built artifacts
(`forge build --root contracts`). Without them, every chain cell skips politely and tells
you what to install. **Estimated time:** ~45 minutes.

## 1. The cast list — what is real tonight, what is an understudy

An honest playbill before the curtain. **Infra mode: `anvil` (guarded)** — the
maximum-realism configuration that still runs with no lab and no LLM endpoint: real chain,
real signing, real controller; a recording double for the router; a scripted stand-in for
the language model.

| Role | Played by tonight | Real? | Taught in | To make it fully real |
|---|---|---|---|---|
| the chain | a disposable Anvil, clock pinned to story time | **REAL** | 06 | it already is — a public chain adds only gas prices and block times |
| settlement + ticket mint | `A2ASettlement` + `MockTOK`, freshly deployed | **REAL** | 06 | — |
| the signatures (EIP-712 offer, EIP-191 proof) | `ChainClient`s holding real keys | **REAL** | 07 | — |
| the bouncer (predicate · auth · translators · state machine) | the production controller, composed by `build_runtime` | **REAL** | 08 | — |
| the hands (gNMI → router) | `MockProvisioner`, the recording double | *understudy* | 09 | `containerlab deploy -t netlab/topology.clab.yml`, then swap in `GnmiProvisioner` (see 09 and `console_explore`) |
| the judgment (Ada's accept/reject) | the **real** `decide()` + retry loop, fed by a scripted stub LLM | *machinery real, words scripted* | 10 | set `LLM_BASE_URL` / `LLM_MODEL` / `LLM_API_KEY` (ADR-001) |
| all of it, in a browser | — | — | — | `just console` — the operator SPA over this exact pipeline |

Everything else — every shape, port, check, and state transition — is the production code
path, unmodified. First, the preflight: two booleans decide whether tonight's chain cells
perform or skip (`chainmcp.testing`'s guard pair, the pattern every chain notebook uses).

In [ ]:
from pathlib import Path

from chainmcp.testing import anvil_available, artifacts_available

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

CHAIN_OK = anvil_available() and artifacts_available()
SKIP = "⏭ skipped: needs anvil + forge artifacts — install Foundry, then `forge build --root contracts`"

print("anvil on PATH:      ", "✓" if anvil_available() else "✗  (install Foundry: https://getfoundry.sh)")
print("forge artifacts:    ", "✓" if artifacts_available() else "✗  (run: forge build --root contracts)")
print("performing tonight: ", "the full play ✓" if CHAIN_OK else "understudies only — chain cells will skip")

Shared props next: the canonical fixtures (the one source of truth you dissected in 04)
and two one-line narration helpers — `hm()` renders unix seconds as the story clock, and
`beat()` prints each step tagged with the **trust domain** acting at that moment. 00
mapped trust domains package by package — nine of them; tonight's play cares about *who
acts*, so its tags group the cast into four runtime lanes — `agent` / `chain` /
`controller` / `network` — plus a plain `story` tag for stage directions. Watch the tags
as the play runs: they tell you *who* you are trusting at every moment.

In [ ]:
import datetime
import time

import a2a_interfaces.fixtures as fx
from a2a_interfaces import EntitlementReader, ErrorCode, NetworkProvisioner, SessionState

TOK = 10**18                                       # 1 TOK in base units (04: integer money)
fmt = lambda wei: f"{wei / TOK:g} TOK"


def hm(ts: int) -> str:
    return datetime.datetime.fromtimestamp(ts, datetime.UTC).strftime("%H:%M")


def beat(domain: str, msg: str) -> None:
    print(f"[{domain:>10}] {msg}")


print("Tonight's canonical cast (a2a_interfaces.fixtures — change it there or not at all):")
beat("story", f"Ada  {fx.ADA}")
beat("story", f"Bell {fx.BELL}")
beat("story", f"ticket #{fx.TICKET_ID} · {fx.CAPACITY_50_MBPS / 1e6:g} Mbps · "
              f"{int(fx.PRICE_10_TOK) / TOK:g} TOK · window {hm(fx.WINDOW.start)}–{hm(fx.WINDOW.end)}")
assert (fx.TICKET_ID, fx.CAPACITY_50_MBPS, fx.PRICE_10_TOK) == (7, 50_000_000, "10000000000000000000")

## 2. Curtain up — a disposable world, frozen at 13:32 · `[chain]` · taught in 06

`launch_anvil` starts a throwaway chain whose clock is **pinned to story time**:
`WINDOW.start − 1680` = 13:32, twenty-eight minutes before Ada's window opens. Chain time
is the only clock that counts (ADR-004) — your wall clock says 2026, the chain says
15 Sep 2025, and only the chain's opinion matters. The deploy order (MockTOK first, from
account #0) makes the token land exactly on the address the fixtures promised — check it.

In [ ]:
if CHAIN_OK:
    from chainmcp.testing import launch_anvil

    STORY_TIME = fx.WINDOW.start - 1680           # 13:32 — before the window, on purpose
    anvil = launch_anvil(timestamp=STORY_TIME)

    print("rpc:       ", anvil.rpc_url)
    print("deployment:", anvil.deployment)
    assert anvil.deployment["MockTOK"] == fx.MOCK_TOK   # deploy order kept the fixtures' promise
    beat("chain", f"a fresh world exists, and its clock says {hm(STORY_TIME)}")
else:
    print(SKIP)

Identity = client (rule 2, taught in 07): one `ChainClient` per key, and the key never
leaves the object. Ada and Bell you know; **Carol** is tonight's stage crew — she exists
to buy Bell's first six tickets so that Ada's mint is *literally* serial #7, the way the
story tells it. `poll_interval=0.2` makes the revocation watcher snappy for act four.

In [ ]:
if CHAIN_OK:
    from chainmcp import ChainClient
    from chainmcp.testing import ANVIL_KEYS

    make = lambda key: ChainClient(anvil.rpc_url, key, deployment=anvil.deployment, poll_interval=0.2)
    ada, bell, carol = make(ANVIL_KEYS["ada"]), make(ANVIL_KEYS["bell"]), make(ANVIL_KEYS["carol"])

    assert (ada.address, bell.address) == (fx.ADA, fx.BELL)    # the cast IS the fixtures' cast
    print("Ada   =", ada.address)
    print("Bell  =", bell.address)
    print("Carol =", carol.address, " ← stage crew")
    print("chain_time →", hm(ada.chain_time()), "(story time, not your wall clock — ADR-004)")
else:
    print(SKIP)

Money: the faucet mints lab TOK freely (scarcity is not tonight's plot; the *mechanics*
of payment are). Ada gets 50 TOK — five times the price. Carol gets exactly 60: six
presale tickets at 10 TOK each, to the last wei.

In [ ]:
if CHAIN_OK:
    ada.faucet(50 * TOK)
    carol.faucet(60 * TOK)

    for name, who in [("Ada", ada.address), ("Bell", bell.address), ("Carol", carol.address)]:
        print(f"{name:6} {fmt(ada.tok_balance(who)):>8}")
    assert ada.tok_balance(bell.address) == 0     # Bell earns his TOK on stage, or not at all
else:
    print(SKIP)

**The presale** (the trick from `chain_client_explore`): ERC-721 serials are mint order,
nothing more (04). Six purchases before Ada's make the next serial 7. Each presale needs a
**fresh salt and a fresh signature** — offers are single-use (invariant I2), so Bell signs
six more little promises. Watch the serials count up.

In [ ]:
if CHAIN_OK:
    for i in range(1, 7):
        presale = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + f"{i:064x}"})
        _, eid = carol.approve_and_fulfill(bell.sign_offer(presale))
        print(f"ticket #{eid} → Carol   (salt …{i:04x}, freshly signed)")
    assert eid == 6                                # the next mint is LITERALLY #7
    beat("chain", "six sold — serial #7 is next")
else:
    print(SKIP)

**✏️ Your turn 2.1 — Carol's receipts.** Six tickets at 10 TOK each, from a 60 TOK
faucet. Write your prediction of Carol's remaining balance as a comment, predict whose
address `owner_of(3)` returns, then run and un-comment the self-checks. Success: both
asserts pass and neither number surprises you.

In [ ]:
if CHAIN_OK:
    # My prediction: Carol has __ TOK left, and ticket #3 belongs to ______
    print("Carol's balance →", fmt(carol.tok_balance(carol.address)))
    print("owner_of(3)     →", carol.owner_of(3))
    # TODO: un-comment once you've written your prediction above:
    # assert carol.tok_balance(carol.address) == 0
    # assert carol.owner_of(3) == carol.address
else:
    print(SKIP)

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
assert carol.tok_balance(carol.address) == 0        # 60 − 6 × 10, to the last wei
assert carol.owner_of(3) == carol.address           # she bought serials 1–6, so #3 is hers
```

Integer base units make the arithmetic exact — 60 TOK in, six 10-TOK pulls out, zero
left, no rounding anywhere (04).
</details>

## 3. Act one — the deal · `[agent]` · taught in 07 (the signature) + 10 (the judgment)

Bell quotes the canonical offer and signs it **EIP-712** — off-chain, no transaction, no
gas. The 65-byte signature *is* the promise. We verify locally that it recovers to Bell's
address over the exact digest the contract will recompute (the cross-stack seam you
proved byte-for-byte in 07 — here just one assert to confirm it still holds).

In [ ]:
if CHAIN_OK:
    from eth_account import Account

    from chainmcp import encode_offer

    signed = bell.sign_offer(fx.CANONICAL_OFFER, terms_doc=fx.TERMS_DOC)
    print("signature:", signed.signature[:26], "…", signed.signature[-6:])
    assert len(bytes.fromhex(signed.signature[2:])) == 65        # the whole promise is 65 bytes

    recovered = Account.recover_message(
        encode_offer(fx.CANONICAL_OFFER, anvil.deployment["chainId"], anvil.deployment["A2ASettlement"]),
        signature=signed.signature,
    )
    assert recovered == bell.address                             # the 07 seam, still sound
    beat("agent", "Bell's promise exists — off-chain, no gas, nothing on the chain yet")
else:
    print(SKIP)

Now the **only judgment call in tonight's play**. Rule 1 gives LLM judgment exactly two
homes: the provider's quote/decline and the consumer's accept/reject — nothing else in
the system may judge, and the controller *never* does. Tonight the provider slot is
scripted (Bell simply quotes the canonical offer — we just signed it), and Ada's consumer
slot performs **for real**: the production `agents.decision.decide()` with its
validate-and-retry loop, fed by an understudy LLM.

The understudy is the exact pattern the repo's own tests use
(`agents/tests/test_llm_retry.py`): a scripted chat object replaces the network endpoint
*inside* a real `LLMClient` — everything above the wire (prompt assembly, schema
validation, retries, the safe decline) is production code.

In [ ]:
from agents.llm import LLMClient, LLMConfig


class ScriptedChat:
    """The agents/tests pattern: replay fixed assistant replies, one per call."""

    def __init__(self, replies):
        self._replies = list(replies)
        self.calls = 0

    def create(self, **_kwargs):          # the surface openai's chat.completions exposes
        reply = self._replies[self.calls]
        self.calls += 1
        return type("R", (), {"choices": [type("C", (), {"message": type("M", (), {"content": reply})})]})


def stub_llm(replies):
    """A real LLMClient whose network half is swapped for a script — retry loop intact."""
    client = LLMClient(LLMConfig(base_url="stub", model="stub", api_key="stub", max_retries=3))
    client._client = type("O", (), {"chat": type("Ch", (), {"completions": ScriptedChat(replies)})})()
    return client


print("understudy ready — it says exactly what we script, through the REAL client")

Run the decision. We script the understudy to misbehave first — a prose reply (not JSON),
then JSON missing the required `reason` — so you can watch the retry loop earn its keep
before the third reply validates. Be clear about what's real here: the **words** of the
reason are ours (a stub can't judge); the **machinery** — prompt assembly from the need +
offer + budget, schema validation against `DecisionOutput`, the retries, the safe decline
— is exactly what runs against a live model. And note: judgment needs no chain — if the
chain cells skipped, the fixtures' placeholder-signed offer stands in.

In [ ]:
from agents.decision import decide

offer_on_the_table = signed if CHAIN_OK else fx.CANONICAL_SIGNED_OFFER   # judgment needs no chain

llm = stub_llm([
    "Sure! Let me think about whether this offer is any good...",         # prose — not JSON
    '{"accept": true}',                                                   # JSON — but no reason
    '{"accept": true, "reason": "50 Mbps in the right window for 10 TOK, under the 15 TOK budget"}',
])
verdict = decide(llm, fx.BANDWIDTH_NEED, offer_on_the_table, budget_tok=15)

print("attempts:", llm.last_attempts, " ← two bad replies burned, retried in code")
print("accept:  ", verdict.accept)
print("reason:  ", verdict.reason)
assert verdict.accept is True and llm.last_attempts == 3
beat("agent", "Ada accepts — the ONLY judgment call in tonight's play (rule 1)")

**✏️ Your turn 3.1 — exhaust the retries.** The retry budget is 3. Make *all three*
replies garbage and re-run `decide`. Success: no exception reaches you — the verdict is
the **safe decline**, `accept=False` with the "declining" reason (a procurement agent
that can't read an offer must not accept it).

In [ ]:
replies = [
    "hmm, let me think",
    "the offer looks fine to me!",
    '{"accept": true, "reason": "fits the need and the budget"}',   # TODO: break this one too
]
exhausted = stub_llm(replies)
cautious = decide(exhausted, fx.BANDWIDTH_NEED, offer_on_the_table, budget_tok=15)
print("accept:", cautious.accept, "| reason:", cautious.reason)
# TODO: un-comment once all three replies are junk:
# assert cautious.accept is False and "declining" in cautious.reason

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
replies = ["hmm, let me think", "the offer looks fine to me!", "still just words"]
exhausted = stub_llm(replies)
cautious = decide(exhausted, fx.BANDWIDTH_NEED, offer_on_the_table, budget_tok=15)
assert cautious.accept is False and "declining" in cautious.reason
```

`client.structured` raises `StructuredError` after three failed validations, and
`decide` catches it and returns the safe decline — the graph never sees a hallucinated
shape and never crashes on one (10).
</details>

## 4. Act two — the atomic swap · `[chain]` · taught in 06 + 07

Ada acts on her acceptance: `approve_and_fulfill` grants an allowance for **exactly**
10 TOK, then calls `fulfill`, which pulls the payment and mints the ticket in **one
transaction** — either everything happens or nothing does (invariant I3). We assert every
effect: Ada −10, Bell +10, ticket #7 exists, its owner is Ada, and the offer's digest is
consumed.

In [ ]:
if CHAIN_OK:
    before = {"ada": ada.tok_balance(ada.address), "bell": ada.tok_balance(bell.address)}

    tx_hash, ticket_id = ada.approve_and_fulfill(signed)

    print(f"tx {tx_hash[:20]}… minted ticket #{ticket_id}")
    assert ticket_id == fx.TICKET_ID == 7                                # the story, literal
    assert ada.owner_of(7) == fx.ADA                                     # ticket → Ada
    assert before["ada"] - ada.tok_balance(ada.address) == 10 * TOK      # Ada  −10 TOK
    assert ada.tok_balance(bell.address) - before["bell"] == 10 * TOK    # Bell +10 TOK
    assert ada.offer_consumed(fx.CANONICAL_OFFER)                        # digest punched (I2)
    view = ada.get(7)
    assert view.params.capacity_bps == 50_000_000 and not view.revoked
    beat("chain", "one transaction: payment pulled AND ticket minted — or neither (I3)")
else:
    print(SKIP)

Try the replay. Same signature, same salt, second submission — the contract already
punched this offer's digest, so it refuses by name. The names travel from Solidity
through the ABI decoder into Python as `ChainRevert.name` (06), the same names the
cardboard `FakeChain` raised in 05 — one failure surface, two realities.

In [ ]:
if CHAIN_OK:
    from chainmcp import ChainRevert

    caught = None
    try:
        ada.approve_and_fulfill(signed)            # same signed offer, second try
    except ChainRevert as err:
        caught = err
    assert caught is not None and caught.name == "OfferAlreadyUsed"
    print("replay →", caught.name, " ← the contract names its refusal")
else:
    print(SKIP)

**✏️ Your turn 4.1 — the no-trace check.** The replay *failed* — but `approve` had
already granted a fresh 10 TOK allowance before `fulfill` refused. Predict (as comments):
Ada's balance now, and the *standing allowance* the settlement still holds over her TOK.
Then check. Success: a rejected purchase left no trace — hint: re-read
`approve_and_fulfill`'s `except ChainRevert` branch in `chainmcp/client.py` (07).

In [ ]:
if CHAIN_OK:
    # My prediction: Ada's balance is __ TOK, and the standing allowance is __
    settlement = anvil.deployment["A2ASettlement"]
    allowance = ada._tok.functions.allowance(ada.address, settlement).call()   # _private: peeking to learn
    print("Ada now   →", fmt(ada.tok_balance(ada.address)))
    print("allowance →", allowance)
    # TODO: un-comment once you've predicted:
    # assert ada.tok_balance(ada.address) == 40 * TOK
    # assert allowance == 0
else:
    print(SKIP)

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
assert ada.tok_balance(ada.address) == 40 * TOK     # 50 − 10 from the ONE real purchase
assert allowance == 0                               # the refused attempt withdrew its own approve
```

The contract's atomicity (I3) covers `fulfill`, but the `approve` was its own mined
transaction — so the client withdraws the allowance itself when `fulfill` refuses. A
rejected purchase truly leaves no trace.
</details>

## 5. Act three — the door · `[controller]` · taught in 08

Time to assemble the bouncer — through the **real production seam**, not by hand:
`controller.wiring.build_runtime` composes a keyless `ChainReader` (the controller holds
no key, rule 2), the `AuthStore`, the translators, and the checked-in resource map
(ADR-005: the ONE file where an opaque on-chain `resourceId` meets a device name) around
`ControllerService`. We inject our `MockProvisioner` as the hands.
`start_watchers` then arms the two autonomous teardown paths you'll see in acts four and
five: the `Revoked`-event watcher and the expiry timer.

In [ ]:
if CHAIN_OK:
    from controller.service import Denied
    from controller.wiring import build_runtime
    from netctl.mock import MockProvisioner

    net = MockProvisioner()                        # the understudy hands (09)
    runtime = build_runtime(anvil.rpc_url, net, deployment=anvil.deployment, poll_interval=0.2)
    svc = runtime.service
    runtime.start_watchers(expiry_poll_s=0.2)      # revocation watcher + expiry timer, armed

    reader = runtime._reader                       # _private: peeking to learn
    assert isinstance(reader, EntitlementReader) and isinstance(net, NetworkProvisioner)   # rule 7
    assert [n for n in dir(reader) if "sign" in n or "fulfill" in n or "faucet" in n] == []
    beat("controller", "door manned: keyless reader ✓  auth ✓  translators ✓  hands ✓")
else:
    print(SKIP)

Before the happy path, watch the bouncer **bounce**. It is 13:32 — Ada owns ticket #7,
her proof will verify, and she is still refused, because the window opens at 14:00. The
dance (08): `challenge()` mints a single-use nonce → Ada's key signs the EIP-191 proof
string → `activate()` verifies the proof, *then* runs the predicate — which answers
`E_NOT_STARTED`. One deny, properly earned.

In [ ]:
if CHAIN_OK:
    ch = svc.challenge(7)
    print("challenge:", ch.nonce[:14] + "…", " expires", hm(ch.expires_at))
    sig, _ = ada.sign_activation_proof(7, ch.nonce, ch.controller_id, ch.expires_at)

    denied = None
    try:
        svc.activate(7, "bandwidth", ch.nonce, sig)
    except Denied as err:
        denied = err
    assert denied is not None and denied.code == ErrorCode.E_NOT_STARTED
    print("activate at", hm(ada.chain_time()), "→ Denied:", denied.code.value,
          " ← window opens", hm(fx.WINDOW.start))
    assert svc._sessions == {}                     # a denial provisions nothing
else:
    print(SKIP)

Now move the world to 14:02. Not `time.sleep(1800)` — **`evm_increaseTime`**: we warp
the *chain's* clock, because chain time is the only clock the system honors (ADR-004).
This is the lab's superpower and production's forbidden fruit: on a public chain you
would simply wait.

In [ ]:
if CHAIN_OK:
    anvil.increase_time(ada._w3, (fx.WINDOW.start + 120) - ada.chain_time())   # _w3: lab-only time control
    now = ada.chain_time()
    print("chain_time →", hm(now), " (we moved the WORLD's clock, not our watch)")
    assert fx.WINDOW.start <= now < fx.WINDOW.end
else:
    print(SKIP)

Same knock, in-window — but first, proof that the failed attempt **cost something**: the
old nonce burned on the attempt (any attempt, even a denied one — no free probing, 08).
Replaying the 13:32 proof now answers `E_NONCE_REUSED`. So: fresh challenge, fresh proof,
and this time all three beats pass — proof verifies, predicate says `None` (allowed),
translator output is applied through the port.

In [ ]:
if CHAIN_OK:
    stale = None
    try:
        svc.activate(7, "bandwidth", ch.nonce, sig)      # the pre-14:00 proof, replayed
    except Denied as err:
        stale = err
    assert stale is not None and stale.code == ErrorCode.E_NONCE_REUSED
    print("old nonce   →", stale.code.value, " ← burned on the FAILED attempt")

    ch2 = svc.challenge(7)                                # fresh nonce, fresh expiry
    sig2, _ = ada.sign_activation_proof(7, ch2.nonce, ch2.controller_id, ch2.expires_at)
    info = svc.activate(7, "bandwidth", ch2.nonce, sig2)
    sid = info.session_id

    assert info.state == SessionState.ACTIVE
    print("fresh nonce →", info.state.value, "  session:", sid)
    beat("controller", "proof verified → predicate 6/6 → translated → applied: the door opens")
else:
    print(SKIP)

Inspect the applied call · `[network]` · taught in 09. The mock recorded **exactly** what
a router would have been told — the same kwargs, byte-for-byte, that the golden-pinned
translator produces (08) and that `GnmiProvisioner` turns into a gNMI `Set` on `srl1`
(09). The ticket's opaque 32-byte `resource_id` became a concrete device + interface pair
through the resource map (ADR-005).

In [ ]:
if CHAIN_OK:
    cfg = net.applied[sid]
    print("what a router would have been told:")
    print("  method       → apply_bandwidth")
    print("  path         →", cfg["path"])
    print("  capacity_bps →", cfg["capacity_bps"], " ← the 50 Mbps on the ticket, now an intent")
    print("  qos_class    →", cfg["qos_class"])
    assert cfg["capacity_bps"] == 50_000_000 and cfg["path"].device == "srl1"
else:
    print(SKIP)

**✏️ Your turn 5.1 — knock about a ticket that doesn't exist.** Ask the controller for a
challenge for ticket **#99**. Predict the `ErrorCode` first (write it as a comment), then
run. Success: the error name tells you the controller checked the *chain* before even
minting a nonce.

In [ ]:
if CHAIN_OK:
    # My prediction: the ErrorCode will be E____________
    probe = None
    try:
        svc.challenge(99)                 # TODO: try 8 too — reserved in the story, not minted tonight (yet)
    except Denied as err:
        probe = err
    print("challenge(99) →", probe.code.value if probe else "no denial!?")
    # TODO: un-comment once you've predicted:
    # assert probe is not None and probe.code == ErrorCode.E_UNKNOWN_ENTITLEMENT
else:
    print(SKIP)

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
assert probe is not None and probe.code == ErrorCode.E_UNKNOWN_ENTITLEMENT
```

`challenge()` calls `reader.get(99)` first; the contract's `ERC721NonexistentToken`
revert becomes a `KeyError` (port parity with the fake — 07), which the service maps to
`E_UNKNOWN_ENTITLEMENT`. Unknown tickets don't even get a nonce.
</details>

## 6. Act four — the showpiece · `[chain → controller → network]`

The sentence this whole repo exists to make true: **Bell revokes ticket #7 on-chain,
mid-window, and the session dies — nobody calls the controller.** The chain of custody:
the `revoke` transaction mines → the contract emits `Revoked(7)` → the reader's watcher
thread (polling every 0.2 s) sees the event → it calls `handle_revoked`, which
**re-reads the chain** before acting (never trust the event — the chain is the source of
truth) → teardown → the mock's records empty.

In [ ]:
if CHAIN_OK:
    beat("chain", "mid-window: Bell pulls the flag on ticket #7")
    bell.revoke(7)

    deadline = time.monotonic() + 5
    while svc.session(sid).state == SessionState.ACTIVE and time.monotonic() < deadline:
        time.sleep(0.05)                           # the watcher polls every 0.2 s — give it a beat

    assert ada.get(7).revoked                                  # the chain says so…
    assert svc.session(sid).state == SessionState.TORN_DOWN    # …the session is gone…
    assert sid not in net.applied and sid in net.torn_down     # …and so is the "policer"
    beat("network", "the throughput died because a flag flipped on a blockchain")
else:
    print(SKIP)

**Rule 8, demonstrated:** teardown is idempotent — calling it again (and again) is a
success, not an error. The watcher already tore this session down once; we tear it down
twice more by hand. Every call lands in `net.torn_down` (idempotence is *observable*,
not just claimed), and nothing is left applied.

In [ ]:
if CHAIN_OK:
    again = svc.teardown(sid)
    once_more = svc.teardown(sid)
    assert again.state == once_more.state == SessionState.TORN_DOWN   # rule 8: still a success
    assert net.applied == {}                                          # nothing lingers
    print("teardown × 3 →", once_more.state.value, " (the watcher's + ours + ours)")
    print("net.torn_down →", net.torn_down, " ← every call recorded")
else:
    print(SKIP)

**✏️ Your turn 6.1 — the ghost session.** Tear down a session that *never existed*:
`svc.teardown("never-existed")`. Predict the returned state and what `net.torn_down`
records, then check. Success: rule 8 extends even to ghosts — and the mock shows the
belt-and-braces sweep.

In [ ]:
if CHAIN_OK:
    # My prediction: state == ____________, and net.torn_down's last entry is ____________
    ghost = svc.teardown("never-existed")
    print("ghost session →", ghost.state.value)
    print("net.torn_down →", net.torn_down[-2:])
    # TODO: un-comment once you've predicted:
    # assert ghost.state == SessionState.TORN_DOWN
    # assert net.torn_down[-1] == "never-existed"
else:
    print(SKIP)

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
assert ghost.state == SessionState.TORN_DOWN
assert net.torn_down[-1] == "never-existed"
```

Unknown sessions answer `torn_down`, and the service still sweeps the provisioner by
name — if some stray config *did* exist under that id, it would be gone now. Belt and
braces (08).
</details>

## 7. The alternate ending — the clock runs out · `[controller]`

The *other* autonomous teardown: expiry. Ticket #7 is revoked and its offer spent, so we
stage an encore: same terms, **fresh salt** → a new ticket. (It will mint as serial #8 —
in the canonical story that serial belongs to Bell's telemetry ticket, but serials are
mint order, not meaning; tonight the serial goes to a bandwidth encore.) Activate it,
warp the world to exactly 16:00, and the `ExpiryTimer` ends it: the OS thread only
*schedules* wake-ups — every wake re-reads **chain time** before tearing anything down
(ADR-004, in code, in `controller/wiring.py`).

In [ ]:
if CHAIN_OK:
    encore = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + f"{0xE0C0:064x}"})
    _, encore_id = ada.approve_and_fulfill(bell.sign_offer(encore))
    print("encore ticket: #" + str(encore_id), " ← same terms, fresh salt (I2)")
    assert encore_id == 8

    ch3 = svc.challenge(encore_id)
    sig3, _ = ada.sign_activation_proof(encore_id, ch3.nonce, ch3.controller_id, ch3.expires_at)
    info3 = svc.activate(encore_id, "bandwidth", ch3.nonce, sig3)
    assert info3.state == SessionState.ACTIVE

    anvil.increase_time(ada._w3, fx.WINDOW.end - ada.chain_time())    # 16:00:00 — [start, end) is over
    deadline = time.monotonic() + 5
    while svc.session(info3.session_id).state == SessionState.ACTIVE and time.monotonic() < deadline:
        time.sleep(0.05)

    assert svc.session(info3.session_id).state == SessionState.TORN_DOWN
    assert net.applied == {}
    print("at", hm(ada.chain_time()), "the ExpiryTimer noticed — no revocation, no request, just the clock")
else:
    print(SKIP)

**✏️ Your turn 7.1 — tick by hand.** The timer thread already ended everything, but you
can call the same method it calls: `svc.tick()`. Predict its return value (write it as a
comment), then run. Success: your prediction matches, and you can say *why* in one
sentence.

In [ ]:
if CHAIN_OK:
    # My prediction: svc.tick() returns ____________
    ended = svc.tick()
    print("tick() →", ended)
    # TODO: un-comment once you've predicted:
    # assert ended == []
else:
    print(SKIP)

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
assert ended == []
```

`tick()` re-reads chain time and ends sessions that are ACTIVE past their `expires_at` —
there are none left, so it returns an empty list. Ticking an already-quiet world is a
no-op, not an error: the same idempotence story as teardown (rule 8).
</details>

## 8. Curtain call — an existence proof is not yet evidence

What you just watched ran **once**, with two understudies. That is an *existence proof*:
the architecture can perform the story. It is **not** yet an answer to "is this
feasible?" — that needs definitions, boundaries, and distributions instead of anecdotes.

The repo has already done that work against the fully-real stack (real router, real LLM),
and the numbers are worth quoting here as a teaser (from
[`docs/evidence/M6.2.md`](../../../docs/evidence/M6.2.md) and
[`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md)):

- **the real hands:** on the SR Linux lab, the demo's iperf shows *100 M offered →
  49.1 Mbps policed*, and full rate back after teardown — tonight's `net.applied` dict,
  made physical
- **speed:** deterministic end-to-end lifecycle median **89 ms**; the predicate alone,
  isolated, **86 ns** per allow (denials 51–83 ns)
- **the showpiece, measured:** revocation lag ≈ the watcher's poll interval + an ~80 ms
  actuation floor — a tunable knob, not an architectural cost
- **adversarial:** 12/12 attacks rejected, each at its predicted layer (3 by the
  contract, 9 by the controller)
- **money:** `fulfill` costs 268k gas (bandwidth) / 447k (telemetry); the real LLM judged
  12/12 decisions correctly against ground truth

How those numbers were *earned* — the seven skeptic's questions, the two definitions of
"enforced", the five simulation boundaries, the harness — is notebook **13**. What they
*mean*, with every boundary attached, is notebook **14**.

And when you want tonight's play with all understudies replaced: bring up the lab and run
[`console_explore.ipynb`](../console_explore.ipynb) or `just console` (the browser SPA
over this exact pipeline), or `uv run python -m e2e.demo` for the narrated
real-router run.

## 9. The integration exam — twelve questions, no cells

You have now seen everything the course builds. Answer each question *out loud* before
unfolding — each is tagged with the notebook that taught it. If one stumps you, that tag
is your reading assignment.

**Q1 · [00+12]** 00 mapped nine package-level trust domains; tonight grouped the cast
into four runtime lanes. Name the four, and for each border the play crossed, name the
artifact that crossed it.
<details><summary>Answer</summary>
`agent` / `chain` / `controller` / `network`. Crossings: a `SignedOffer` went
agent → chain (`fulfill`); an `EntitlementView` went chain → controller (the reader);
an `apply_bandwidth(...)` intent went controller → network (the provisioner port); the
`Revoked(7)` event went chain → controller (the watcher).
</details>

**Q2 · [02]** Why would `signed.offer.price = "5"` never work — what property of every
cross-package shape prevents it, and what would tampering do to the offer anyway?
<details><summary>Answer</summary>
All `a2a_interfaces` shapes are frozen pydantic models — mutation raises a
`ValidationError`. You'd need `model_copy(update=...)`, which makes a *new* object — and
any changed field makes Bell's signature recover to a stranger, so `fulfill` reverts
`BadSignature`.
</details>

**Q3 · [03]** `MockProvisioner` never inherits from `NetworkProvisioner`, yet
`isinstance(net, NetworkProvisioner)` said `True` tonight. What's the mechanism, and why
does the whole architecture bet on it?
<details><summary>Answer</summary>
Structural typing: `NetworkProvisioner` is a `@runtime_checkable` `typing.Protocol` —
having the right methods *is* satisfying it. That's what lets fakes and real adapters
swap under an unchanged controller (and rule 7 makes them behaviorally interchangeable
via the shared contract tests).
</details>

**Q4 · [04]** Why is the price the *string* `"10000000000000000000"` and not the float
`10.0`?
<details><summary>Answer</summary>
Money is integer base units (1 TOK = 10¹⁸) so on-chain arithmetic is exact; the wire
shape uses a decimal **string** because JSON numbers are floats and would corrupt 256-bit
integers.
</details>

**Q5 · [06]** Tonight's chain booted with `--timestamp` pinned to 13:32. Why must the
notebook control the *chain's* clock rather than just… run at whatever time it is?
<details><summary>Answer</summary>
ADR-004: `block.timestamp` is the only validity clock. The canonical offer and window are
absolute unix times in Sep 2025 — on a chain that thinks it's 2026, the offer is long
expired. Pinning the timestamp makes chain time story time.
</details>

**Q6 · [07]** Two different signatures appeared tonight — Bell's EIP-712 offer and Ada's
EIP-191 activation proof. What does each prove, and who verifies each, where?
<details><summary>Answer</summary>
EIP-712: the provider's binding promise over twelve typed fields — verified by the
*contract* (`hashOffer` + `ecrecover`) at `fulfill`. EIP-191: live possession of the
ticket-owning key — verified by the *controller* in Python
(`Account.recover_message` vs `ownerOf`). Chain verifies the sale; controller verifies
the door.
</details>

**Q7 · [08]** The first knock failed `E_NOT_STARTED`, yet replaying the same proof later
gave `E_NONCE_REUSED`. Why does a *failed* activation still consume the challenge?
<details><summary>Answer</summary>
The nonce burns on ANY verification attempt, success or failure — otherwise a stolen
proof (or a probing attacker) could retry against the same challenge forever. A failed
attempt costs a challenge; ask for a fresh one.
</details>

**Q8 · [09]** Tonight's "policer" was a dict entry. What makes that a meaningful
stand-in — which rule, and what enforces it in CI?
<details><summary>Answer</summary>
Rule 7: mocks implement the same Protocol as real adapters and pass the SAME shared
contract-test suite (`netctl/tests/test_provisioner_contract.py`). The recorded kwargs
are exactly what `GnmiProvisioner` would send to `srl1` — pinned further by the golden
translator files.
</details>

**Q9 · [10]** Rule 1 gives LLM judgment exactly two homes. Name both, say which one
performed tonight, and name the guard that keeps a wandering model from crashing the
system.
<details><summary>Answer</summary>
The consumer's accept/reject (`decide` — performed tonight, on a stub) and the provider's
quote/decline (scripted tonight). The guard is the validate-and-retry loop: the caller
gets a schema-valid `DecisionOutput` or a safe decline — never a raw string. And the
controller is never an LLM.
</details>

**Q10 · [11]** Between 05's cardboard play and tonight, what changed — and what provably
did *not*?
<details><summary>Answer</summary>
The adapters changed: `FakeChain` → Anvil + `ChainClient`s/`ChainReader`, placeholder
signature → real EIP-712, `advance()` → `evm_increaseTime`. The lifecycle script, the two
ports, and every line of controller logic did not — that's the composition-root swap 11
demonstrated.
</details>

**Q11 · [12]** Trace `bell.revoke(7)` to `net.applied` emptying — every hop, including
the one deliberate re-check along the way.
<details><summary>Answer</summary>
Revoke tx mines → contract emits `Revoked(7)` → the reader's watcher thread polls new
blocks and sees the log → calls `handle_revoked(7)` → the service **re-reads**
`get(7).revoked` from the chain (never trusts the event) → `teardown(session)` → the
provisioner removes the config (`applied.pop`) → session state `TORN_DOWN`.
</details>

**Q12 · [13, next]** Why is tonight's green run not yet an answer to "is this
architecture feasible"?
<details><summary>Answer</summary>
It's n=1 with two understudies and no measurements: no definitions of "enforced", no
stated simulation boundaries, no distributions, no adversarial coverage. Feasibility is a
measured claim — which is exactly where the course goes next.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| preflight + guarded chain cells | an honest cast list: real vs understudy, with upgrade recipes | the five simulation boundaries, formalized (13) |
| pinned Anvil + Carol's presale | deterministic stage dressing — chain time is story time, serials are mint order | `e2e/tests/conftest.py` seeds every chain-profile test run the same way |
| real `decide()` on a scripted LLM | judgment *machinery* vs judgment *words* (rule 1) | E5 measures the real model's judgment (13/14) |
| `approve_and_fulfill` + full-effect asserts | the atomic swap (I2, I3), replay refused by name | per-service gas costs in TOK and dollars (14) |
| `build_runtime` + challenge–response | the production seam; possession proven live; nonces burn on attempt | `just up` serves this same service over HTTP (M6.1) |
| the revocation showpiece | event-driven teardown that re-checks the chain | the revocation-lag sweep E9 (13/14) |
| the encore + `ExpiryTimer` | ADR-004: threads schedule wake-ups, the chain decides | the expiry experiment's 73 ms median (14) |

## Experiments to try

Predict first, then run:

- **Tamper:** re-sign a fresh-salt offer, then flip one hex character of the signature
  before `approve_and_fulfill`. Predict which layer refuses, by what name, and whether
  any TOK moves. (Then re-run exercise 4.1's checks.)
- **Impostor:** in act three, have **Bell** sign the activation proof
  (`bell.sign_activation_proof(...)`) for a fresh challenge on ticket #7. Predict the
  `ErrorCode` — owning the issuing key is not owning the ticket.
- **Slow watcher:** restart the kernel and run with `expiry_poll_s=2.0` (and
  `poll_interval=1.0`). Predict how much later act four and the encore land — then meet
  the same curve, measured, in notebook 14's revocation-lag sweep.
- **Deliberate breakage:** comment out `runtime.start_watchers(...)` and replay act
  four. Predict what happens after `bell.revoke(7)` — and what that says about which
  component *is* the enforcement channel.

## Check yourself

The exam above **is** the check — but three quick ones to close the night:

1. Which two roles were understudies tonight, and what is the one-line recipe that makes
   each fully real?
2. The first activation attempt was denied — what did it cost Ada, and why is that cost a
   security feature?
3. Two sessions ended tonight without anyone calling the controller. Name both triggers
   and the clock each one consulted.

**Where this goes next:** [`13_the_evaluation.ipynb`](13_the_evaluation.ipynb) — the play
ran once; now we ask the skeptic's seven questions and turn it into defensible numbers.

**Deeper dives:** [`console_explore.ipynb`](../console_explore.ipynb) (this pipeline with
the real router, driven by the operator console) ·
[`e2e/src/e2e/demo.py`](../../src/e2e/demo.py) (the narrated real-router demo) ·
[`docs/evidence/M6.2.md`](../../../docs/evidence/M6.2.md) +
[`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md) (the numbers quoted above, with
their provenance) · [`docs/00-the-story.md`](../../../docs/00-the-story.md) (the epilogue
you just performed).

One last ritual — strike the set. Always fold the world away: stop the controller's
threads, the clients' watchers, and the chain itself.

In [ ]:
if CHAIN_OK:
    runtime.close()                     # stops the ExpiryTimer, closes the reader's watcher
    for client in (ada, bell, carol):
        client.close()
    anvil.stop()
    print("world folded away — house lights up. Rerun from the top anytime.")
else:
    print("nothing to fold — the chain never launched. Install Foundry + `forge build --root contracts` to see the play.")